In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/MLProject/dataset-test"
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# SMILES DATASET

In [ ]:
from __future__ import annotations

import argparse
import os
import time
from typing import Any

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

OUT_DIR = DATA_DIR
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data"

TARGETS = {
    "GLP1R": "CHEMBL1784",
    "GIPR": "CHEMBL2035",
    "GCGR": "CHEMBL3272",
}


def safe_get_json(
    url: str,
    params: dict[str, Any] | None = None,
    timeout: int = 45,
    retries: int = 3,
    sleep_s: float = 1.0,
) -> dict[str, Any] | None:
    """GET JSON with light retry logic for Colab/network instability."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as exc:
            if attempt == retries:
                print(f"[WARN] GET failed: {url} params={params} error={exc}")
                return None
            time.sleep(sleep_s * attempt)
    return None


def fetch_bioactivity(
    target_id: str,
    target_name: str,
    limit: int = 1000,
    sleep_s: float = 0.25,
) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    offset = 0

    while True:
        params = {
            "target_chembl_id": target_id,
            "standard_type__in": "EC50,IC50,Ki,Kd,AC50,Emax",
            "format": "json",
            "limit": limit,
            "offset": offset,
        }
        data = safe_get_json(f"{BASE_URL}/activity.json", params=params)
        if not data:
            break

        batch = data.get("activities", [])
        if not batch:
            break

        for row in batch:
            row["target_name"] = target_name
            row["target_chembl_id_query"] = target_id
        records.extend(batch)

        total = data.get("page_meta", {}).get("total_count", len(records))
        print(f"{target_name}: {min(len(records), total)}/{total}", end="\r")
        offset += limit
        if offset >= total:
            break
        time.sleep(sleep_s)

    print()
    return pd.DataFrame(records)


def extract_sequence_from_molecule_json(data: dict[str, Any]) -> str | None:
    """
    Extract peptide/protein sequence from a ChEMBL molecule JSON object.

    ChEMBL stores sequence-like information in several possible shapes.
    The most useful case for peptide drugs is usually:
        data["biotherapeutic"]["biocomponents"][i]["sequence"]
    """
    seqs: list[str] = []

    # Case 1: biotherapeutic components, common for peptide/protein therapeutics.
    bio = data.get("biotherapeutic")
    if isinstance(bio, dict):
        comps = bio.get("biocomponents")
        if isinstance(comps, list):
            for comp in comps:
                if not isinstance(comp, dict):
                    continue
                seq = comp.get("sequence")
                if isinstance(seq, str) and seq.strip():
                    seqs.append(seq.strip())

    if seqs:
        # Multiple chains/components are separated by '.', similar to multi-part molecules.
        return ".".join(seqs)

    # Case 2: direct sequence-like fields, kept as fallback.
    for key in ["sequence", "molecule_sequence", "protein_sequence"]:
        seq = data.get(key)
        if isinstance(seq, str) and seq.strip():
            return seq.strip()

    return None


def fetch_molecule_sequence(
    mol_chembl_id: str,
    sleep_s: float = 0.05,
) -> str | None:
    """Fetch peptide/protein sequence for one ChEMBL molecule ID, if available."""
    if not isinstance(mol_chembl_id, str) or not mol_chembl_id:
        return None

    url = f"{BASE_URL}/molecule/{mol_chembl_id}.json"
    data = safe_get_json(url)
    time.sleep(sleep_s)

    if not data:
        return None

    return extract_sequence_from_molecule_json(data)


def add_sequences(
    df: pd.DataFrame,
    sleep_s: float = 0.05,
) -> pd.DataFrame:
    """Add a sequence column by querying unique molecule_chembl_id values."""
    if df.empty or "molecule_chembl_id" not in df.columns:
        df = df.copy()
        df["sequence"] = np.nan
        return df

    df = df.copy()
    unique_mols = df["molecule_chembl_id"].dropna().astype(str).unique()

    print(f"\nFetching peptide sequences for {len(unique_mols):,} unique molecules...")
    seq_map: dict[str, str | None] = {}

    for mol_id in tqdm(unique_mols, desc="ChEMBL sequence fetch"):
        seq_map[mol_id] = fetch_molecule_sequence(mol_id, sleep_s=sleep_s)

    df["sequence"] = df["molecule_chembl_id"].astype(str).map(seq_map)
    return df


def classify_assay(description: Any) -> str:
    d = str(description or "").lower()
    if any(k in d for k in ["camp", "cyclic amp", "adenylyl", "adenylate", "gs-coupled", "cre-luc"]):
        return "cAMP"
    if any(k in d for k in ["arrestin", "pathhunter", "bret", "tango"]):
        return "beta_arrestin"
    if any(k in d for k in ["binding", "radioligand", "displacement"]):
        return "binding"
    if any(k in d for k in ["insulin", "glucagon", "glucose", "weight"]):
        return "functional"
    return "other"


def value_to_pactivity(value: Any, units: Any) -> float:
    try:
        v = float(value)
    except Exception:
        return np.nan

    unit_map = {
        "pm": 1e-12,
        "nm": 1e-9,
        "um": 1e-6,
        "µm": 1e-6,
        "mm": 1e-3,
        "m": 1.0,
    }
    factor = unit_map.get(str(units or "").strip().lower())
    if factor is None or v <= 0:
        return np.nan
    return -np.log10(v * factor)


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()

    desc_col = "assay_description" if "assay_description" in df.columns else "description"
    df["assay_class"] = df[desc_col].apply(classify_assay) if desc_col in df.columns else "other"
    df["pactivity"] = df.apply(
        lambda r: value_to_pactivity(r.get("standard_value"), r.get("standard_units")),
        axis=1,
    )
    df = df[df["pactivity"].between(4, 13)]

    # Keep activity relation because values such as ">" or "<" are censored and
    # should not be treated as exact efficacy labels downstream.
    if "standard_relation" not in df.columns:
        df["standard_relation"] = np.nan

    keep = [
        "molecule_chembl_id",
        "molecule_pref_name",
        "canonical_smiles",
        "sequence",
        "target_name",
        "target_chembl_id_query",
        "standard_type",
        "standard_relation",
        "standard_value",
        "standard_units",
        "pactivity",
        "assay_class",
        "assay_chembl_id",
        "assay_description",
        "document_chembl_id",
        "src_id",
        "bao_endpoint",
        "bao_format",
    ]
    keep = [c for c in keep if c in df.columns]
    df = df[keep + [c for c in df.columns if c not in keep]]

    dedup = [
        c
        for c in [
            "molecule_chembl_id",
            "target_name",
            "assay_chembl_id",
            "standard_type",
            "standard_value",
        ]
        if c in df.columns
    ]
    if dedup:
        df = df.drop_duplicates(subset=dedup)

    return df.reset_index(drop=True)


def molecule_summary(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    group_cols = ["molecule_chembl_id", "target_name", "assay_class"]
    med = (
        df.groupby(group_cols)["pactivity"]
        .median()
        .reset_index()
        .pivot_table(
            index=["molecule_chembl_id", "target_name"],
            columns="assay_class",
            values="pactivity",
            aggfunc="median",
        )
        .reset_index()
    )
    med.columns.name = None

    agg_dict: dict[str, Any] = {
        "n_activity": ("pactivity", "size"),
    }
    if "molecule_pref_name" in df.columns:
        agg_dict["molecule_pref_name"] = ("molecule_pref_name", "first")
    if "canonical_smiles" in df.columns:
        agg_dict["canonical_smiles"] = ("canonical_smiles", "first")
    if "sequence" in df.columns:
        agg_dict["sequence"] = ("sequence", "first")

    mol_info = df.groupby("molecule_chembl_id").agg(**agg_dict).reset_index()

    out = med.merge(mol_info, on="molecule_chembl_id", how="left")
    if "cAMP" in out.columns and "beta_arrestin" in out.columns:
        out["bias_ratio_cAMP_minus_barr"] = out["cAMP"] - out["beta_arrestin"]

    return out


def make_morgan_fp_csv(
    df_mol: pd.DataFrame,
    out_path: str,
    n_bits: int = 2048,
    radius: int = 2,
) -> None:
    try:
        from rdkit import Chem
        from rdkit.Chem import AllChem
    except Exception as exc:
        print(f"[WARN] RDKit unavailable; skipping fingerprints. Install rdkit-pypi. error={exc}")
        return

    rows = []
    for r in tqdm(df_mol.itertuples(index=False), total=len(df_mol), desc="Morgan FP"):
        smi = getattr(r, "canonical_smiles", None)
        mol_id = getattr(r, "molecule_chembl_id")
        if not isinstance(smi, str) or not smi:
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
        arr = np.zeros((n_bits,), dtype=np.int8)
        for idx in fp.GetOnBits():
            arr[idx] = 1
        row = {"molecule_chembl_id": mol_id}
        row.update({f"fp_{i}": int(arr[i]) for i in range(n_bits)})
        rows.append(row)

    pd.DataFrame(rows).to_csv(out_path, index=False)
    print(f"Saved: {out_path}")


def write_report(df: pd.DataFrame, df_mol: pd.DataFrame, out_path: str) -> None:
    lines = []
    lines.append("GLP-1/GIP/GCGR ChEMBL background collection report")
    lines.append("=" * 62)
    lines.append(f"activity rows: {len(df):,}")
    lines.append(f"molecule-target summary rows: {len(df_mol):,}")
    lines.append("")

    lines.append("[Target distribution]")
    lines.append(df["target_name"].value_counts(dropna=False).to_string() if not df.empty else "empty")
    lines.append("")

    lines.append("[Assay class distribution]")
    lines.append(df["assay_class"].value_counts(dropna=False).to_string() if not df.empty else "empty")
    lines.append("")

    if not df.empty:
        n_smiles_activity = df["canonical_smiles"].notna().sum() if "canonical_smiles" in df.columns else 0
        n_seq_activity = df["sequence"].notna().sum() if "sequence" in df.columns else 0
        lines.append(f"activity rows with SMILES: {n_smiles_activity:,}")
        lines.append(f"activity rows with sequence: {n_seq_activity:,}")
        lines.append("")

    if not df_mol.empty:
        n_smiles = df_mol["canonical_smiles"].notna().sum() if "canonical_smiles" in df_mol.columns else 0
        n_seq = df_mol["sequence"].notna().sum() if "sequence" in df_mol.columns else 0
        lines.append(f"molecule-target rows with SMILES: {n_smiles:,}")
        lines.append(f"molecule-target rows with sequence: {n_seq:,}")
        if "bias_ratio_cAMP_minus_barr" in df_mol.columns:
            n_bias = df_mol["bias_ratio_cAMP_minus_barr"].notna().sum()
            lines.append(f"molecule-target rows with cAMP and beta-arrestin: {n_bias:,}")

    text = "\n".join(lines)
    print(text)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--out_dir", default=OUT_DIR)
    parser.add_argument(
        "--make_fp",
        type=int,
        default=0,
        help="Set 1 to write Morgan fingerprints if RDKit is installed.",
    )
    parser.add_argument(
        "--fetch_sequence",
        type=int,
        default=1,
        help="Set 1 to fetch peptide/protein sequences from ChEMBL molecule endpoint.",
    )
    parser.add_argument(
        "--sequence_sleep_s",
        type=float,
        default=0.05,
        help="Sleep time between molecule sequence requests.",
    )

    # Colab/Jupyter injects kernel arguments such as
    # "-f /root/.local/share/jupyter/runtime/kernel-....json".
    # parse_known_args keeps the script usable both as `!python file.py` and
    # from notebook cells that call main() directly.
    args, _unknown = parser.parse_known_args()

    os.makedirs(args.out_dir, exist_ok=True)

    all_dfs = []
    for target_name, target_id in TARGETS.items():
        print(f"\nCollecting {target_name} ({target_id})")
        raw = fetch_bioactivity(target_id, target_name)
        clean = preprocess(raw)
        print(f"{target_name}: raw={len(raw):,}, clean={len(clean):,}")
        all_dfs.append(clean)

    df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

    if args.fetch_sequence:
        df = add_sequences(df, sleep_s=args.sequence_sleep_s)
        # Reorder columns after sequence insertion.
        df = preprocess(df)
    elif "sequence" not in df.columns:
        df["sequence"] = np.nan

    activity_path = os.path.join(args.out_dir, "background_chembl_activity.csv")
    df.to_csv(activity_path, index=False)
    print(f"Saved: {activity_path}")

    df_mol = molecule_summary(df)
    summary_path = os.path.join(args.out_dir, "background_chembl_molecule_summary.csv")
    df_mol.to_csv(summary_path, index=False)
    print(f"Saved: {summary_path}")

    report_path = os.path.join(args.out_dir, "background_chembl_report.txt")
    write_report(df, df_mol, report_path)

    if args.make_fp:
        fp_path = os.path.join(args.out_dir, "background_chembl_morgan_fp.csv")
        make_morgan_fp_csv(df_mol, fp_path)


if __name__ == "__main__":
    main()



GLP1R: 2775/2775
GLP1R: raw=2,775, clean=2,478

GIPR: 3866/3866
GIPR: raw=3,866, clean=2,053

GCGR: 226/226
GCGR: raw=226, clean=217

Fetching peptide sequences for 3,534 unique molecules...


ChEMBL sequence fetch:   0%|          | 0/3534 [00:00<?, ?it/s]

Saved: ./glp1_data/background_chembl_activity.csv
Saved: ./glp1_data/background_chembl_molecule_summary.csv
GLP-1/GIP/GCGR ChEMBL background collection report
activity rows: 4,748
molecule-target summary rows: 3,537

[Target distribution]
target_name
GLP1R    2478
GIPR     2053
GCGR      217

[Assay class distribution]
assay_class
cAMP             1739
binding          1626
other            1179
beta_arrestin     166
functional         38

activity rows with SMILES: 4,724
activity rows with sequence: 34

molecule-target rows with SMILES: 3,531
molecule-target rows with sequence: 4
molecule-target rows with cAMP and beta-arrestin: 67


# Sequence DATASET

In [ ]:
"""
채널:
  1. UniProt REST API
  2. Ensembl REST API       (GCG/GIP 오솔로그)
  3. NCBI protein DB
  4. NCBI nucleotide CDS   (mRNA → CDS 번역)
  5. RCSB PDB              (구조 결정된 펩타이드 리간드)
"""

import time, json, csv, re, urllib.request, urllib.parse
from collections import Counter, defaultdict

OUT_FILE = "Sequences.csv"
DELAY    = 0.4
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

ENSEMBL  = "https://rest.ensembl.org"
UNIPROT  = "https://rest.uniprot.org/uniprotkb/search"
ESEARCH  = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
EFETCH   = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
RCSB_SEARCH     = "https://search.rcsb.org/rcsbsearch/v2/query"
RCSB_DATA       = "https://data.rcsb.org/rest/v1/core"
RCSB_FASTA      = "https://www.rcsb.org/fasta/entry/{pdb_id}/download"

# ── 모티프 ────────────────────────────────────────────────────────────
GCG_MOTIFS  = ["HSQGTFT","HSQGTFS","HAQGTFT","HSQATFT","HSEGTFT",
               "HAQGTFS","HTEGTFT","HSKGTFT","HSQSTFT","HSQGTLT"]
GLP1_MOTIFS = ["HAEGTFT","HSEGTFT","HAEGTFS","HADGTFT","HAEGTFA",
               "HGEGTFT","HAEGTYT","HAEGTLT","HAEGTIT","HAEGTIS",
               "HTEGTFT","HADGTLT","HGEGTFS","HAEGTTT","HAEATIS"]
GIP_MOTIFS  = ["YAEGTFI","YAEQFIS","HAEGTFI","YAEGTFS","YAEQFTS",
               "YAEGTLT","YAEGT","YAEGTFT","YVEGTFI","YAEGTFV",
               "YAEQFSS","YAEEFIS","YAEGTLS"]
EXENDIN_MOTIFS = ["HGEGTFT","HSDGTFT","HGEGSFT"]
GCG_LEN, GLP1_LEN, GIP_LEN = 29, 30, 42


def find_motif(seq, motifs):
    for m in motifs:
        idx = seq.find(m)
        if idx >= 0:
            return idx, m
    return -1, None


def extract_mature(seq, family):
    """모티프 기반 mature peptide 추출. fallback 없음."""
    seq = seq.upper().strip()
    results = []
    fam = family.upper()

    is_gcg     = any(k in fam for k in ["GCG","PROGLUCAGON","GLUCAGON","GLP1","GLP-1"])
    is_gip     = any(k in fam for k in ["GIP","GASTRIC","INSULINOTROPIC"])
    is_exendin = "EXENDIN" in fam

    if is_exendin:
        if 20 <= len(seq) <= 80 and not seq.startswith("M"):
            results.append(("Exendin", seq, "direct"))
        return results

    if is_gcg:
        idx, m = find_motif(seq, GCG_MOTIFS)
        if idx >= 0:
            pep = seq[idx:idx+GCG_LEN]
            if 25 <= len(pep) <= 33:
                results.append(("Glucagon_ortholog", pep, m))
        idx, m = find_motif(seq, GLP1_MOTIFS)
        if idx >= 0:
            pep = seq[idx:idx+GLP1_LEN]
            if 25 <= len(pep) <= 37:
                results.append(("GLP1_ortholog", pep, m))

    if is_gip:
        idx, m = find_motif(seq, GIP_MOTIFS)
        if idx >= 0:
            pep = seq[idx:idx+GIP_LEN]
            if 38 <= len(pep) <= 45:
                results.append(("GIP_ortholog", pep, m))

    return results


def aa_quality_ok(seq, max_bad=0.15):
    if not seq:
        return False
    bad = sum(1 for c in seq if c not in STANDARD_AA)
    return bad / len(seq) <= max_bad


def make_row(source, label, acc, org, seq, motif, notes):
    seq = seq.upper().strip()
    return {
        "source": source, "label": label,
        "accession": acc, "organism": org,
        "sequence": seq, "length": len(seq),
        "motif": motif,
        "cAMP_pEC50": "", "barr_pEC50": "", "bias_ratio": "",
        "notes": notes,
    }


def process_entry(source, family, acc, org, seq, notes):
    rows = []
    seq = seq.upper().strip()
    if not seq or acc.lower().startswith("pdb"):
        return rows
    for label, pep, motif in extract_mature(seq, family):
        if not aa_quality_ok(pep):
            continue
        rows.append(make_row(source, label, acc, org, pep, motif, notes))
    return rows


# ══════════════════════════════════════════════════════════════════════
# 채널 1: UniProt
# ══════════════════════════════════════════════════════════════════════
TAXA = {
    "mammals": "40674", "birds": "8782", "reptiles": "8504",
    "amphibians": "8292", "fish": "7898",
}

def build_uniprot_queries():
    q = []
    for taxon, tid in TAXA.items():
        q.append((f"GCG_{taxon}",
                  f"gene:GCG AND taxonomy_id:{tid} AND length:[90 TO 240]",
                  1000, "GCG"))
        q.append((f"GIP_{taxon}",
                  f"gene:GIP AND taxonomy_id:{tid} AND length:[60 TO 240]",
                  1000, "GIP"))
    q += [
        ("GCG_name",
         "(protein_name:proglucagon OR protein_name:glucagon) "
         "AND taxonomy_id:7742 AND length:[25 TO 240]",
         1000, "GCG"),
        ("GLP1_name",
         "(protein_name:\"glucagon-like peptide-1\" "
         "OR protein_name:\"glucagon-like peptide 1\") "
         "AND taxonomy_id:7742 AND length:[20 TO 120]",
         1000, "GLP1"),
        ("GIP_name",
         "(protein_name:\"gastric inhibitory polypeptide\" "
         "OR protein_name:\"glucose-dependent insulinotropic polypeptide\") "
         "AND taxonomy_id:7742 AND length:[25 TO 240]",
         1000, "GIP"),
        ("Exendin_all",
         "protein_name:exendin AND length:[20 TO 80]",
         500, "EXENDIN"),
        ("Glucagon_mature",
         "protein_name:glucagon AND taxonomy_id:7742 AND length:[25 TO 40]",
         500, "GCG"),
    ]
    return q


def fetch_uniprot(label, query, max_results, family):
    print(f"\n[UniProt] {label}", flush=True)
    params = urllib.parse.urlencode({
        "query": query, "format": "tsv",
        "fields": "accession,gene_names,organism_name,sequence,length",
        "size": min(max_results, 500),
    })
    url, rows, page = f"{UNIPROT}?{params}", [], 0
    while url and len(rows) < max_results:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "academic/final"})
            with urllib.request.urlopen(req, timeout=25) as r:
                text = r.read().decode("utf-8")
                link = r.headers.get("Link", "")
            next_url = None
            for part in link.split(","):
                if 'rel="next"' in part:
                    next_url = part.split(";")[0].strip().strip("<>")
            lines = text.strip().split("\n")
            if page == 0 and lines:
                lines = lines[1:]
            for line in lines:
                cols = line.split("\t")
                if len(cols) < 4:
                    continue
                acc, gene, org, seq = cols[0], cols[1], cols[2], cols[3]
                if any(x in gene.upper() for x in ["GLP1R","GIPR","GCGR"]):
                    continue
                rows.extend(process_entry("UniProt", family, acc, org, seq, label))
            page += 1
            url = next_url
            print(f"  page {page}: {len(rows)}건", flush=True)
            time.sleep(DELAY)
        except Exception as e:
            print(f"  ERROR: {e}")
            break
    print(f"  → {len(rows)}건")
    return rows


# ══════════════════════════════════════════════════════════════════════
# 채널 2: Ensembl (배치 10, timeout 60s)
# ══════════════════════════════════════════════════════════════════════
def ensembl_get(endpoint, params=None):
    url = f"{ENSEMBL}{endpoint}"
    if params:
        url += "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={
        "Content-Type": "application/json",
        "Accept": "application/json",
        "User-Agent": "academic/final"
    })
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())


def get_ensembl_orthologs(gene_symbol):
    print(f"\n[Ensembl] Homology: {gene_symbol}", flush=True)
    try:
        data = ensembl_get(
            f"/homology/symbol/human/{gene_symbol}",
            {"type": "orthologues", "format": "full",
             "sequence": "protein", "aligned": 0}
        )
    except Exception as e:
        print(f"  ERROR: {e}")
        return []
    orthologs = []
    for item in data.get("data", [{}])[0].get("homologies", []):
        target = item.get("target", {})
        pid = target.get("protein_id", "")
        sp  = target.get("species", "")
        if pid:
            orthologs.append((pid, sp))
    print(f"  → {len(orthologs)}개 protein_id")
    return orthologs


def fetch_ensembl_sequences(orthologs, family, batch_size=10):
    rows, total = [], len(orthologs)
    print(f"[Ensembl] {family} 서열 fetch ({total}개)", flush=True)
    for i in range(0, total, batch_size):
        batch  = orthologs[i:i+batch_size]
        pids   = [pid for pid, sp in batch]
        id2sp  = {pid: sp.replace("_", " ").capitalize() for pid, sp in batch}
        payload = json.dumps({"ids": pids}).encode()
        req = urllib.request.Request(
            f"{ENSEMBL}/sequence/id",
            data=payload,
            headers={"Content-Type": "application/json",
                     "Accept": "application/json",
                     "User-Agent": "academic/final"},
            method="POST"
        )
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                seqs = json.loads(r.read())
            for entry in seqs:
                pid = entry.get("id", "")
                seq = entry.get("seq", "")
                org = id2sp.get(pid, "")
                rows.extend(process_entry(
                    "Ensembl", family, pid, org, seq,
                    f"Ensembl ortholog {family}"))
        except Exception as e:
            print(f"  Batch {i//batch_size+1} ERROR: {e}")
        if (i//batch_size+1) % 5 == 0:
            print(f"  batch {i//batch_size+1}/{(total-1)//batch_size+1}: "
                  f"{len(rows)}건", flush=True)
        time.sleep(DELAY)
    print(f"  → {len(rows)}건")
    return rows


# ══════════════════════════════════════════════════════════════════════
# 채널 3: NCBI protein
# ══════════════════════════════════════════════════════════════════════
NCBI_PROTEIN_QUERIES = [
    ("GCG_ncbi_mammals",
     "proglucagon[Protein Name] AND Mammalia[Organism] AND 90:240[Sequence Length]",
     800, "GCG"),
    ("GCG_ncbi_birds",
     "proglucagon[Protein Name] AND Aves[Organism] AND 90:240[Sequence Length]",
     400, "GCG"),
    ("GCG_ncbi_fish",
     "proglucagon[Protein Name] AND Actinopterygii[Organism] AND 90:240[Sequence Length]",
     400, "GCG"),
    ("GCG_ncbi_amphibia",
     "proglucagon[Protein Name] AND Amphibia[Organism] AND 90:240[Sequence Length]",
     300, "GCG"),
    ("GCG_ncbi_reptiles",
     "proglucagon[Protein Name] AND Reptilia[Organism] AND 90:240[Sequence Length]",
     300, "GCG"),
    ("GIP_ncbi_all",
     "(\"gastric inhibitory polypeptide\"[Protein Name] "
     "OR \"glucose-dependent insulinotropic polypeptide\"[Protein Name] "
     "OR GIP[Gene Name]) AND Vertebrata[Organism] AND 25:240[Sequence Length]",
     1000, "GIP"),
    ("GLP1_ncbi_mature",
     "(\"glucagon-like peptide 1\"[Protein Name] "
     "OR \"glucagon-like peptide-1\"[Protein Name]) "
     "AND Vertebrata[Organism] AND 20:120[Sequence Length]",
     800, "GLP1"),
    ("Exendin_ncbi",
     "exendin[Protein Name] AND 20:80[Sequence Length]",
     300, "EXENDIN"),
    ("Glucagon_ncbi_mature",
     "glucagon[Protein Name] AND Vertebrata[Organism] AND 25:40[Sequence Length]",
     500, "GCG"),
]


def fetch_ncbi_protein(label, query, max_results, family):
    print(f"\n[NCBI protein] {label}", flush=True)
    params = urllib.parse.urlencode({
        "db": "protein", "term": query,
        "retmax": max_results, "retmode": "json",
    })
    try:
        with urllib.request.urlopen(f"{ESEARCH}?{params}", timeout=20) as r:
            data  = json.loads(r.read())
        ids   = data["esearchresult"]["idlist"]
        total = data["esearchresult"]["count"]
        print(f"  total={total}, fetch={len(ids)}")
    except Exception as e:
        print(f"  Search ERROR: {e}")
        return []
    rows = []
    for i in range(0, len(ids), 100):
        batch = ids[i:i+100]
        fp = urllib.parse.urlencode({
            "db": "protein", "id": ",".join(batch),
            "rettype": "fasta", "retmode": "text",
        })
        try:
            with urllib.request.urlopen(f"{EFETCH}?{fp}", timeout=30) as r:
                fasta = r.read().decode("utf-8")
            cur_hdr, cur_seq, entries = "", [], []
            for line in fasta.strip().split("\n"):
                if line.startswith(">"):
                    if cur_hdr and cur_seq:
                        entries.append((cur_hdr, "".join(cur_seq)))
                    cur_hdr, cur_seq = line[1:], []
                else:
                    cur_seq.append(line.strip())
            if cur_hdr and cur_seq:
                entries.append((cur_hdr, "".join(cur_seq)))
            for hdr, seq in entries:
                org_m = re.search(r"\[(.+?)\]$", hdr)
                org   = org_m.group(1) if org_m else ""
                acc   = hdr.split()[0]
                if acc.lower().startswith("pdb"):
                    continue
                rows.extend(process_entry("NCBI_protein", family, acc, org, seq, label))
            print(f"  batch {i//100+1}: {len(rows)}건", flush=True)
        except Exception as e:
            print(f"  Fetch ERROR: {e}")
        time.sleep(DELAY)
    print(f"  → {len(rows)}건")
    return rows


# ══════════════════════════════════════════════════════════════════════
# 채널 4: NCBI nucleotide CDS
# ══════════════════════════════════════════════════════════════════════
NCBI_NUC_QUERIES = [
    ("GCG_mRNA_mammals",
     "GCG[gene] AND Mammalia[orgn] AND mRNA[filter]", 300, "GCG"),
    ("GCG_mRNA_birds",
     "GCG[gene] AND Aves[orgn] AND mRNA[filter]", 200, "GCG"),
    ("GCG_mRNA_fish",
     "GCG[gene] AND Actinopterygii[orgn] AND mRNA[filter]", 200, "GCG"),
    ("GCG_mRNA_reptiles",
     "GCG[gene] AND Reptilia[orgn] AND mRNA[filter]", 150, "GCG"),
    ("GCG_mRNA_amphibia",
     "GCG[gene] AND Amphibia[orgn] AND mRNA[filter]", 150, "GCG"),
    ("GIP_mRNA_mammals",
     "GIP[gene] AND Mammalia[orgn] AND mRNA[filter]", 300, "GIP"),
    ("GIP_mRNA_birds",
     "GIP[gene] AND Aves[orgn] AND mRNA[filter]", 150, "GIP"),
]


def fetch_ncbi_cds(label, query, max_results, family):
    print(f"\n[NCBI nuc] {label}", flush=True)
    params = urllib.parse.urlencode({
        "db": "nucleotide", "term": query,
        "retmax": max_results, "retmode": "json",
    })
    try:
        with urllib.request.urlopen(f"{ESEARCH}?{params}", timeout=20) as r:
            data  = json.loads(r.read())
        ids   = data["esearchresult"]["idlist"]
        total = data["esearchresult"]["count"]
        print(f"  total={total}, fetch={len(ids)}")
    except Exception as e:
        print(f"  Search ERROR: {e}")
        return []
    rows = []
    for i in range(0, len(ids), 100):
        batch = ids[i:i+100]
        fp = urllib.parse.urlencode({
            "db": "nucleotide", "id": ",".join(batch),
            "rettype": "gp", "retmode": "text",
        })
        try:
            with urllib.request.urlopen(f"{EFETCH}?{fp}", timeout=40) as r:
                text = r.read().decode("utf-8", errors="ignore")
            for rec in text.split("//\n"):
                org_m   = re.search(r'/organism="([^"]+)"', rec)
                org     = org_m.group(1) if org_m else ""
                acc_m   = re.search(r"^ACCESSION\s+(\S+)", rec, re.MULTILINE)
                acc     = acc_m.group(1) if acc_m else ""
                trans_m = re.search(r'/translation="([^"]+)"', rec, re.DOTALL)
                if not trans_m:
                    continue
                trans = re.sub(r"\s+", "", trans_m.group(1))
                rows.extend(process_entry("NCBI_nuc", family, acc, org, trans, label))
            print(f"  batch {i//100+1}: {len(rows)}건", flush=True)
        except Exception as e:
            print(f"  Fetch ERROR: {e}")
        time.sleep(DELAY)
    print(f"  → {len(rows)}건")
    return rows


# ══════════════════════════════════════════════════════════════════════
# 채널 5: RCSB PDB
# ══════════════════════════════════════════════════════════════════════
PDB_SEARCH_TERMS = [
    "GLP-1 receptor", "glucagon-like peptide-1 receptor", "GLP1R",
    "GIP receptor", "GIPR", "glucagon receptor", "GCGR",
    "exendin", "semaglutide", "oxyntomodulin",
    "glucagon-like peptide", "gastric inhibitory polypeptide",
]

PDB_SEED_IDS = {
    "6X18","7KI0","7RTB","9IVM","9C0K","3IOL","3C5T","9EBN",
}

PDB_LABEL_MAP = {
    "PDB_GLP1_like":         "GLP1_pdb",
    "PDB_GLP1_analog":       "GLP1_pdb",
    "PDB_Glucagon_like":     "Glucagon_pdb",
    "PDB_Exendin_like":      "Exendin_pdb",
    "PDB_GIP_like":          "GIP_pdb",
    "PDB_Oxyntomodulin_like":"GLP1_pdb",
}


def pdb_http_get(url, timeout=30):
    req = urllib.request.Request(
        url, headers={"User-Agent": "academic-pdb-collector/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read().decode("utf-8", errors="ignore")


def pdb_http_post(url, payload, timeout=40):
    data = json.dumps(payload).encode()
    req = urllib.request.Request(
        url, data=data,
        headers={"Content-Type": "application/json",
                 "Accept": "application/json",
                 "User-Agent": "academic-pdb-collector/1.0"},
        method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())


def pdb_search_ids(term, rows=200):
    payload = {
        "query": {"type": "terminal", "service": "full_text",
                  "parameters": {"value": term}},
        "return_type": "entry",
        "request_options": {"paginate": {"start": 0, "rows": rows},
                            "results_content_type": ["experimental"]}
    }
    try:
        data = pdb_http_post(RCSB_SEARCH, payload)
        return [x["identifier"] for x in data.get("result_set", [])]
    except Exception as e:
        print(f"  PDB search ERROR [{term}]: {e}")
        return []


def pdb_collect_ids():
    ids = set(PDB_SEED_IDS)
    for term in PDB_SEARCH_TERMS:
        found = pdb_search_ids(term)
        ids.update(x.upper() for x in found)
        time.sleep(DELAY)
    print(f"  PDB candidate entries: {len(ids)}")
    return sorted(ids)


def pdb_infer_label(seq, header):
    h = header.lower()
    if "exendin" in h or "exenatide" in h:
        return "PDB_Exendin_like"
    if "semaglutide" in h or "taspoglutide" in h or "liraglutide" in h:
        return "PDB_GLP1_analog"
    if "oxyntomodulin" in h:
        return "PDB_Oxyntomodulin_like"
    if "glucagon-like peptide" in h or "glp-1" in h or "glp1" in h:
        return "PDB_GLP1_like"
    if "gastric inhibitory" in h or "gip" in h or "insulinotropic" in h:
        return "PDB_GIP_like"
    if "glucagon" in h:
        return "PDB_Glucagon_like"
    # 모티프 기반
    s = seq.upper()
    if any(m in s for m in GIP_MOTIFS):
        return "PDB_GIP_like"
    if any(m in s for m in EXENDIN_MOTIFS):
        return "PDB_Exendin_like"
    if any(m in s for m in GLP1_MOTIFS):
        return "PDB_GLP1_like"
    if any(m in s for m in GCG_MOTIFS):
        return "PDB_Glucagon_like"
    return ""


def pdb_infer_target(meta):
    text = " ".join([
        meta.get("struct", {}).get("title", ""),
        meta.get("struct_keywords", {}).get("pdbx_keywords", ""),
    ]).lower()
    targets = []
    if "glp-1 receptor" in text or "glp1r" in text:
        targets.append("GLP1R")
    if "gip receptor" in text or "gipr" in text:
        targets.append("GIPR")
    if "glucagon receptor" in text or "gcgr" in text:
        targets.append("GCGR")
    return "|".join(sorted(set(targets)))


def pdb_extract_subseq(seq, pdb_label):
    """모티프 기반 서브서열 추출 (PDB chain이 약간 길 때)"""
    s = seq.upper()
    if pdb_label in ("PDB_GIP_like",):
        motifs, length = GIP_MOTIFS, 42
    elif pdb_label in ("PDB_Glucagon_like",):
        motifs, length = GCG_MOTIFS, 29
    elif pdb_label in ("PDB_GLP1_like","PDB_GLP1_analog"):
        motifs, length = GLP1_MOTIFS + EXENDIN_MOTIFS, 30
    elif pdb_label == "PDB_Oxyntomodulin_like":
        motifs, length = GCG_MOTIFS, 37
    else:
        return seq
    for m in motifs:
        idx = s.find(m)
        if idx >= 0:
            pep = s[idx:idx+length]
            if len(pep) >= 20:
                return pep
    return seq


def pdb_process_entry(pdb_id):
    rows = []
    try:
        meta   = json.loads(pdb_http_get(
            f"{RCSB_DATA}/entry/{pdb_id.lower()}"))
        target = pdb_infer_target(meta)
        title  = meta.get("struct", {}).get("title", "")
    except Exception:
        target, title = "", ""

    try:
        fasta_text = pdb_http_get(
            RCSB_FASTA.format(pdb_id=pdb_id.upper()))
    except Exception as e:
        print(f"  FASTA ERROR {pdb_id}: {e}")
        return rows

    # FASTA 파싱
    entries, cur_hdr, cur_seq = [], "", []
    for line in fasta_text.splitlines():
        if line.startswith(">"):
            if cur_hdr and cur_seq:
                entries.append((cur_hdr, "".join(cur_seq).upper()))
            cur_hdr, cur_seq = line[1:], []
        else:
            cur_seq.append(line.strip())
    if cur_hdr and cur_seq:
        entries.append((cur_hdr, "".join(cur_seq).upper()))

    for header, seq in entries:
        # 수용체/G단백질/항체 제거
        h = header.lower()
        bad_words = ["receptor","g protein","guanine nucleotide","arrestin",
                     "nanobody","antibody","fab","heavy chain","light chain",
                     "scfv","lysozyme","fusion"]
        if any(w in h for w in bad_words) or len(seq) > 120:
            continue
        if not (15 <= len(seq) <= 120):
            continue
        if not aa_quality_ok(seq, max_bad=0.10):
            continue

        pdb_label = pdb_infer_label(seq, header)
        if not pdb_label:
            continue

        pep = pdb_extract_subseq(seq, pdb_label)
        if not (15 <= len(pep) <= 80) or not aa_quality_ok(pep, 0.10):
            continue

        # PDB label → unified label
        unified_label = PDB_LABEL_MAP.get(pdb_label, pdb_label)

        # chain 파싱
        chain_m = re.search(r"\|Chains?\s+([^|]+)\|", header)
        chain   = chain_m.group(1).strip() if chain_m else ""

        rows.append({
            "source": "RCSB_PDB",
            "label": unified_label,
            "accession": f"{pdb_id.upper()}_{chain}",
            "organism": "",
            "sequence": pep,
            "length": len(pep),
            "motif": pdb_label,          # 원래 PDB label을 motif 컬럼에 보존
            "cAMP_pEC50": "",
            "barr_pEC50": "",
            "bias_ratio": "",
            "notes": f"PDB:{pdb_id} | target:{target} | {title[:60]}",
        })
    return rows


def fetch_pdb(max_entries=None):
    print("\n[RCSB PDB] ID 수집 중...", flush=True)
    ids = pdb_collect_ids()
    if max_entries:
        ids = ids[:max_entries]
    rows = []
    for i, pdb_id in enumerate(ids, 1):
        if i % 50 == 0:
            print(f"  [{i}/{len(ids)}] {pdb_id} | 누적 {len(rows)}건", flush=True)
        try:
            rows.extend(pdb_process_entry(pdb_id))
        except Exception as e:
            print(f"  ERROR {pdb_id}: {e}")
        time.sleep(DELAY)
    print(f"  → PDB 총 {len(rows)}건")
    return rows


# ══════════════════════════════════════════════════════════════════════
# 중복 제거
# ══════════════════════════════════════════════════════════════════════
VALID_LABELS = {
    "GLP1_ortholog", "GIP_ortholog", "Glucagon_ortholog", "Exendin",
    "GLP1_pdb", "GIP_pdb", "Glucagon_pdb", "Exendin_pdb",
}

def deduplicate(rows):
    seen, out = set(), []
    for r in rows:
        label = r["label"]
        seq   = r["sequence"].upper().strip()
        org   = r.get("organism", "")

        if label not in VALID_LABELS:
            continue
        if not (20 <= len(seq) <= 80):
            continue
        if not aa_quality_ok(seq):
            continue
        # 오솔로그는 organism 필수, PDB는 없어도 OK
        if label.endswith("_ortholog") and not org:
            continue

        key = (label, seq)
        if key not in seen:
            seen.add(key)
            r["sequence"] = seq
            r["length"]   = len(seq)
            out.append(r)
    return out


# ══════════════════════════════════════════════════════════════════════
# 메인
# ══════════════════════════════════════════════════════════════════════
def main():
    all_rows = []

    # ── 채널 1: UniProt ─────────────────────────────────────────────
    print("\n" + "="*55)
    print("채널 1: UniProt")
    print("="*55)
    for label, query, max_r, family in build_uniprot_queries():
        all_rows.extend(fetch_uniprot(label, query, max_r, family))

    # ── 채널 2: Ensembl ─────────────────────────────────────────────
    print("\n" + "="*55)
    print("채널 2: Ensembl")
    print("="*55)
    for gene in ["GCG", "GIP"]:
        orthologs = get_ensembl_orthologs(gene)
        time.sleep(DELAY)
        if orthologs:
            rows = fetch_ensembl_sequences(orthologs, gene, batch_size=10)
            all_rows.extend(rows)
            print(f"  {gene} Ensembl 소계: {len(rows)}건")

    # ── 채널 3: NCBI protein ─────────────────────────────────────────
    print("\n" + "="*55)
    print("채널 3: NCBI protein")
    print("="*55)
    for label, query, max_r, family in NCBI_PROTEIN_QUERIES:
        all_rows.extend(fetch_ncbi_protein(label, query, max_r, family))

    # ── 채널 4: NCBI nucleotide CDS ──────────────────────────────────
    print("\n" + "="*55)
    print("채널 4: NCBI nucleotide CDS")
    print("="*55)
    for label, query, max_r, family in NCBI_NUC_QUERIES:
        all_rows.extend(fetch_ncbi_cds(label, query, max_r, family))

    # ── 채널 5: RCSB PDB ─────────────────────────────────────────────
    print("\n" + "="*55)
    print("채널 5: RCSB PDB")
    print("="*55)
    all_rows.extend(fetch_pdb())

    # ── 중복 제거 ────────────────────────────────────────────────────
    print(f"\n수집 전 총계: {len(all_rows)}건")
    all_rows = deduplicate(all_rows)
    print(f"중복 제거 후:  {len(all_rows)}건")

    if not all_rows:
        print("데이터 없음")
        return

    # ── 통계 ─────────────────────────────────────────────────────────
    print("\n=== Label 통계 ===")
    for k, v in sorted(Counter(r["label"] for r in all_rows).items()):
        print(f"  {k:<25} {v}건")

    print("\n=== Source 통계 ===")
    for k, v in sorted(Counter(r["source"] for r in all_rows).items()):
        print(f"  {k:<20} {v}건")

    print("\n=== 종 다양성 (오솔로그) ===")
    orgs = defaultdict(set)
    for r in all_rows:
        if r["organism"]:
            orgs[r["label"]].add(r["organism"])
    for k, v in sorted(orgs.items()):
        print(f"  {k:<25} {len(v)}종")

    # ── 저장 ─────────────────────────────────────────────────────────
    fieldnames = ["source","label","accession","organism","sequence",
                  "length","motif","cAMP_pEC50","barr_pEC50","bias_ratio","notes"]
    with open(OUT_FILE, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(all_rows)

    print(f"\n✅ {OUT_FILE} ({len(all_rows)}건)")


if __name__ == "__main__":
    main()
